# Notebook 7 — National bridge risk maps

Interactive maps of the Notebook 5 predictions, following `../ma_bridges/bridge_map.ipynb`
(the validated MA visualization): same horizon knob, risk color scale, hover fields, and
carto-positron basemap.

**Inputs** (written by Notebook 5, `train_national_rsf.ipynb`):
- `us_bridge_risk_map.json` — one record per bridge with valid decimal coordinates
  (Notebook 5 already converted the NBI DDMMSS encoding and dropped bad coordinates, so
  the MA notebook's inline DMS-conversion step is not repeated here).
- `us_results_prob_of_failure.csv` — hover context: year built, predicted failure year,
  years remaining, county/place codes, risk category.

**Intentional national deviations from the MA notebook:**
1. **Top-N national view instead of all bridges** — 1.2M hover-enabled points is far
   beyond what a browser renders; the national map shows the `TOP_N` highest-risk
   bridges, and the state view shows every mappable bridge of one state
   (`STATE_FILTER = "25"` reproduces the MA map from the national model).
2. **`px.scatter_map`** replaces `px.scatter_mapbox`, deprecated since plotly 6.
3. **State choropleth added** — a national summary the single-state MA notebook
   had no use for.

In [ ]:
# ── Configuration + load ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

MAP_JSON = Path("us_bridge_risk_map.json")         # decimal coords + 5/10/20-yr probabilities
PROBS    = Path("us_results_prob_of_failure.csv")  # full per-bridge results (hover context)
KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]      # structure # is unique only WITHIN a state

HORIZON       = "prob_fail_10yr"   # 'prob_fail_5yr' / 'prob_fail_10yr' / 'prob_fail_20yr'
HORIZON_LABEL = {"prob_fail_5yr": "5-Year", "prob_fail_10yr": "10-Year",
                 "prob_fail_20yr": "20-Year"}[HORIZON]
TOP_N        = 25_000              # national map size: enough to show every hotspot while
                                   # hover-enabled WebGL rendering stays responsive
STATE_FILTER = "25"                # state drill-down (FIPS); '25' = MA, the validated benchmark

# Same 4-stop scale as the MA map.
RISK_SCALE = [
    [0.0,  "#1D9E75"],   # low risk  - teal
    [0.33, "#EF9F27"],   # medium    - amber
    [0.66, "#E24B4A"],   # high      - red
    [1.0,  "#501313"],   # very high - dark red
]

# dtype= : read_json would otherwise coerce the zero-padded FIPS strings ('02') to integers.
risk = pd.read_json(MAP_JSON, dtype={"STATE_FIPS": str, "STRUCTURE_NUMBER_008": str})

ctx = pd.read_csv(
    PROBS,
    usecols=KEYS + ["YEAR_BUILT_027", "predicted_year_goes_poor", "years_remaining",
                    "COUNTY_CODE_003", "PLACE_CODE_004", "risk_category"],
    dtype={k: str for k in KEYS},
    # pandas 3.0's block-wise parse combined with usecols raises IndexError while
    # composing its mixed-dtype warning (same bug Notebook 4 works around).
    low_memory=False)
# Nullable Int64 keeps NaN-bearing codes readable in hover text ('1957', not '1957.0').
for c in ["YEAR_BUILT_027", "predicted_year_goes_poor", "years_remaining",
          "COUNTY_CODE_003", "PLACE_CODE_004"]:
    ctx[c] = pd.to_numeric(ctx[c], errors="coerce").astype("Int64")

# validate='m:1' guarantees the context join can never duplicate a bridge's map record.
risk = risk.merge(ctx, on=KEYS, how="left", validate="m:1")

def hover_text(d):
    """Same hover fields as the MA map, built only for the rows actually plotted —
    materialising 1.2M hover strings up front would cost ~0.5 GB for nothing."""
    return (
        "Structure: "         + d["STRUCTURE_NUMBER_008"]
        + "<br>State FIPS: "  + d["STATE_FIPS"]
        + "<br>County: "      + d["COUNTY_CODE_003"].astype(str)
        + "<br>Place: "       + d["PLACE_CODE_004"].astype(str)
        + "<br>Year Built: "  + d["YEAR_BUILT_027"].astype(str)
        + "<br>Predicted Year Poor: " + d["predicted_year_goes_poor"].astype(str)
        + "<br>Years Remaining: "     + d["years_remaining"].astype(str)
        + "<br>5yr risk: "    + (d["prob_fail_5yr"]  * 100).round(1).astype(str) + "%"
        + "<br>10yr risk: "   + (d["prob_fail_10yr"] * 100).round(1).astype(str) + "%"
        + "<br>20yr risk: "   + (d["prob_fail_20yr"] * 100).round(1).astype(str) + "%"
        + "<br>Category: "    + d["risk_category"].astype(str)
    )

print(f"{len(risk):,} mappable bridges   horizon: {HORIZON_LABEL}   "
      f"median {HORIZON}: {risk[HORIZON].median():.3f}")

In [ ]:
# ── National view: the TOP_N highest-risk bridges ─────────────────────────────
nat = risk.nlargest(TOP_N, HORIZON).copy()
nat["hover"] = hover_text(nat)

fig = px.scatter_map(
    nat, lat="lat", lon="lon", color=HORIZON,
    color_continuous_scale=RISK_SCALE, range_color=[0, 1],
    zoom=3.2, center={"lat": 38.8, "lon": -96.0},
    map_style="carto-positron",
    title=f"U.S. Bridge Failure Risk — Top {len(nat):,} Bridges by {HORIZON_LABEL} Probability",
    labels={HORIZON: f"{HORIZON_LABEL} Failure Probability"},
    height=700,
)
fig.update_traces(marker=dict(size=4, opacity=0.7),
                  customdata=nat[["hover"]],
                  hovertemplate="%{customdata[0]}<extra></extra>")
fig.update_layout(
    coloraxis_colorbar=dict(title=f"{HORIZON_LABEL}<br>Failure Prob.",
                            tickvals=[0, 0.25, 0.5, 0.75, 1.0],
                            ticktext=["0%", "25%", "50%", "75%", "100%"], len=0.6),
    margin=dict(l=0, r=0, t=50, b=0), font=dict(family="Arial"),
)
fig.show()
# fig.write_html("us_bridge_risk_map_national.html")

In [ ]:
# ── State drill-down: every mappable bridge in STATE_FILTER ───────────────────
st = risk[risk["STATE_FIPS"] == STATE_FILTER].copy()
assert len(st), f"no mappable bridges for STATE_FIPS={STATE_FILTER!r}"
st["hover"] = hover_text(st)

# Web-mercator zoom halves the visible span per level, so log2 of the state's
# coordinate span frames any state (RI to AK) without per-state tuning; the median
# center is robust to outlying territories (e.g. the Aleutians).
lat_span = st["lat"].max() - st["lat"].min()
lon_span = (st["lon"].max() - st["lon"].min()) * 0.75     # longitude shrinks with cos(lat)
zoom = float(np.clip(7.3 - np.log2(max(lat_span, lon_span, 1e-6)), 3.0, 9.0))

fig = px.scatter_map(
    st, lat="lat", lon="lon", color=HORIZON,
    color_continuous_scale=RISK_SCALE, range_color=[0, 1],
    zoom=zoom, center={"lat": st["lat"].median(), "lon": st["lon"].median()},
    map_style="carto-positron",
    title=f"Bridge Failure Risk, FIPS {STATE_FILTER} — {HORIZON_LABEL} Probability "
          f"({len(st):,} bridges)",
    labels={HORIZON: f"{HORIZON_LABEL} Failure Probability"},
    height=700,
)
fig.update_traces(marker=dict(size=6, opacity=0.8),
                  customdata=st[["hover"]],
                  hovertemplate="%{customdata[0]}<extra></extra>")
fig.update_layout(
    coloraxis_colorbar=dict(title=f"{HORIZON_LABEL}<br>Failure Prob.",
                            tickvals=[0, 0.25, 0.5, 0.75, 1.0],
                            ticktext=["0%", "25%", "50%", "75%", "100%"], len=0.6),
    margin=dict(l=0, r=0, t=50, b=0), font=dict(family="Arial"),
)
fig.show()
# fig.write_html(f"us_bridge_risk_map_state_{STATE_FILTER}.html")

In [ ]:
# ── State summary: mean horizon probability per state ─────────────────────────
FIPS_TO_USPS = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA", "08": "CO", "09": "CT",
    "10": "DE", "11": "DC", "12": "FL", "13": "GA", "15": "HI", "16": "ID", "17": "IL",
    "18": "IN", "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME", "24": "MD",
    "25": "MA", "26": "MI", "27": "MN", "28": "MS", "29": "MO", "30": "MT", "31": "NE",
    "32": "NV", "33": "NH", "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI", "45": "SC", "46": "SD",
    "47": "TN", "48": "TX", "49": "UT", "50": "VT", "51": "VA", "53": "WA", "54": "WV",
    "55": "WI", "56": "WY",
}

by_state = (risk.groupby("STATE_FIPS")
                .agg(mean_prob=(HORIZON, "mean"), n_bridges=(HORIZON, "size"))
                .reset_index())
by_state["state"] = by_state["STATE_FIPS"].map(FIPS_TO_USPS)

fig = px.choropleth(
    by_state, locations="state", locationmode="USA-states", scope="usa",
    color="mean_prob", color_continuous_scale=RISK_SCALE,
    # State MEANS sit well below 1, so the color range is data-driven here — the
    # bridge maps' fixed [0, 1] range would wash every state into the low-risk teal.
    hover_data={"n_bridges": ":,", "mean_prob": ":.3f"},
    title=f"Mean {HORIZON_LABEL} Failure Probability by State",
    labels={"mean_prob": f"Mean {HORIZON_LABEL}<br>Failure Prob.", "n_bridges": "Bridges"},
    height=600,
)
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0), font=dict(family="Arial"))
fig.show()

by_state.sort_values("mean_prob", ascending=False).head(10)